# Chapter 3: It Starts with a Tensor — Storage, Strides & Memory Layouts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emreaslan7/ai/blob/main/notebooks/deep-learning-with-pytorch/03-it-starts-with-a-tensor.ipynb)

Interactive workbook for **Chapter 3** of *Deep Learning with PyTorch (2nd Edition, Manning)*.
- **Tensor Foundations:** Unboxed contiguous C arrays, dtypes (`float32`, `bfloat16`, `int64`), multidimensional indexing
- **Broadcasting & Named Tensors:** Dimension expansion rules and semantic dimension tagging
- **Tensor API & In-Place Ops:** Dimensional reductions, `keepdim=True`, and in-place `_` safety
- **Storage & Stride Internals:** 1D flat `UntypedStorage`, offset mapping equation, zero-copy `.t()` / `.permute()`
- **Memory Contiguity:** Non-contiguous layouts, `.is_contiguous()`, `.view()` vs `.contiguous()`
- **Low-Level Stride Tricks:** Sliding window views via `as_strided()`
- **Hardware Acceleration:** CPU $\leftrightarrow$ GPU CUDA transfers, pinned memory, device matching
- **NumPy & Serialization:** Zero-copy CPU buffer sharing, PyTorch checkpoints, chunked HDF5 (`h5py`) storage
- **Chapter Exercises:** Analytical solutions to Section 3.15 storage & view problems

In [2]:
# 0. Dependencies Installation, Core Imports & Device Setup
%pip install -q h5py numpy

import torch
import numpy as np
import h5py
import os

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__} | Default Device: {device}")
if device.type == 'cuda':
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Compute Capability: {torch.cuda.get_device_capability(0)}")

Note: you may need to restart the kernel to use updated packages.PyTorch Version: 2.13.0+cu126 | Default Device: cuda
GPU Model: NVIDIA GeForce GTX 1650
CUDA Compute Capability: (7, 5)

---
## 1. Tensor Foundations & Construction

Create multidimensional tensors and inspect metadata (`shape`, `dtype`, `device`, `numel`, `dim`).

In [4]:
# 1.1 Tensor creation routines
a_1d = torch.tensor([1.0, 2.0, 3.0])
b_2d = torch.ones(3, 2)
c_3d = torch.zeros(2, 4, 4)

print("1D Tensor:", a_1d)
print(f"a_1d -> shape: {a_1d.shape}, dtype: {a_1d.dtype}, elements: {a_1d.numel()}, dim: {a_1d.dim()}")
print(f"b_2d (3x2) shape: {b_2d.shape}")
print(f"c_3d (2x4x4) shape: {c_3d.shape}")

1D Tensor: tensor([1., 2., 3.])
a_1d -> shape: torch.Size([3]), dtype: torch.float32, elements: 3, dim: 1
b_2d (3x2) shape: torch.Size([3, 2])
c_3d (2x4x4) shape: torch.Size([2, 4, 4])

---
## 2. Multidimensional Indexing and Slicing

Access sub-regions, extract rows/columns, and verify negative indexing.

In [5]:
# 2.1 Multidimensional Slicing on a 3x4 Matrix
grid = torch.arange(1, 13, dtype=torch.float32).reshape(3, 4)
print("Original 3x4 Grid:\n", grid)

# Single element scalar extraction
scalar_val = grid[1, 2]
print(f"\ngrid[1, 2] -> {scalar_val.item()} (Python float)")

# Row and Column Slicing
first_col = grid[:, 0]
sub_matrix = grid[1:, 1:3]
print("\nFirst column (grid[:, 0]):", first_col)
print("Sub-matrix (grid[1:, 1:3]):\n", sub_matrix)

Original 3x4 Grid:
 tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.]])

grid[1, 2] -> 7.0 (Python float)

First column (grid[:, 0]): tensor([1., 5., 9.])
Sub-matrix (grid[1:, 1:3]):
 tensor([[ 6.,  7.],
        [10., 11.]])

---
## 3. Broadcasting Mechanics

Perform arithmetic between tensors with compatible singleton dimensions without memory copying.

In [6]:
# 3.1 Broadcast addition between (3, 1) column and (1, 4) row
col_vec = torch.tensor([[10.0], [20.0], [30.0]])   # Shape: (3, 1)
row_vec = torch.tensor([[1.0, 2.0, 3.0, 4.0]])      # Shape: (1, 4)

broadcasted = col_vec + row_vec                     # Broadcasted -> Shape: (3, 4)
print("Column Vector (3, 1):\n", col_vec)
print("Row Vector (1, 4):\n", row_vec)
print("Broadcasted Sum (3, 4):\n", broadcasted)
assert broadcasted.shape == (3, 4), "Broadcasting shape mismatch" 

Column Vector (3, 1):
 tensor([[10.],
        [20.],
        [30.]])
Row Vector (1, 4):
 tensor([[1., 2., 3., 4.]])
Broadcasted Sum (3, 4):
 tensor([[11., 12., 13., 14.],
        [21., 22., 23., 24.],
        [31., 32., 33., 34.]])

---
## 4. Named Tensors

Tag dimensions with explicit identifiers for safety and self-documenting code.

In [18]:
# %pip install einops
import torch
from einops import rearrange

# 1. Tensörü oluştur (NCHW)
imgs = torch.randn(2, 3, 28, 28)

# 2. Önce isimleri belirt, sonra hedef sıralamaya çevir (NCHW -> NHWC)
imgs_reordered = rearrange(imgs, 'batch channels rows cols -> batch rows cols channels')

print("Orijinal Şekil :", imgs.shape)          # torch.Size([2, 3, 28, 28])
print("Yeniden Sıralı :", imgs_reordered.shape)  # torch.Size([2, 28, 28, 3])

Orijinal Şekil : torch.Size([2, 3, 28, 28])
Yeniden Sıralı : torch.Size([2, 28, 28, 3])

---
## 5. Tensor Data Types (`dtype`) & Memory Footprint

Explore float32, bfloat16, int64, and calculate exact RAM consumption.

In [19]:
# 5.1 Inspect dtypes and calculate memory usage
fp32_t = torch.randn(1000, 1000, dtype=torch.float32)
bf16_t = fp32_t.to(dtype=torch.bfloat16)
int64_t = torch.zeros(1000, 1000, dtype=torch.int64)

def get_tensor_memory_mb(t):
    return (t.numel() * t.element_size()) / (1024 * 1024)

print(f"FP32  (1000x1000) -> Element Size: {fp32_t.element_size()}B | Total Memory: {get_tensor_memory_mb(fp32_t):.2f} MB")
print(f"BF16  (1000x1000) -> Element Size: {bf16_t.element_size()}B | Total Memory: {get_tensor_memory_mb(bf16_t):.2f} MB")
print(f"INT64 (1000x1000) -> Element Size: {int64_t.element_size()}B | Total Memory: {get_tensor_memory_mb(int64_t):.2f} MB")

FP32  (1000x1000) -> Element Size: 4B | Total Memory: 3.81 MB
BF16  (1000x1000) -> Element Size: 2B | Total Memory: 1.91 MB
INT64 (1000x1000) -> Element Size: 8B | Total Memory: 7.63 MB

---
## 6. Tensor API: Reductions & In-Place Operations

Dimensional reductions with `dim` / `keepdim=True`, and in-place mutations with `_` suffix.

In [20]:
# 6.1 Dimensional Reductions
mat = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
mean_dim0 = torch.mean(mat, dim=0)
mean_dim1_kept = torch.mean(mat, dim=1, keepdim=True)

print("Original Matrix:\n", mat)
print(f"Mean across dim=0 (Columns): {mean_dim0} | shape={mean_dim0.shape}")
print(f"Mean across dim=1 (Rows, keepdim=True):\n{mean_dim1_kept} | shape={mean_dim1_kept.shape}")

# 6.2 In-Place Operations
x = torch.ones(2, 2)
x.add_(5.0)
print("\nx after x.add_(5.0):\n", x)
x.zero_()
print("x after x.zero_():\n", x)

Original Matrix:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
Mean across dim=0 (Columns): tensor([2.5000, 3.5000, 4.5000]) | shape=torch.Size([3])
Mean across dim=1 (Rows, keepdim=True):
tensor([[2.],
        [5.]]) | shape=torch.Size([2, 1])

x after x.add_(5.0):
 tensor([[6., 6.],
        [6., 6.]])
x after x.zero_():
 tensor([[0., 0.],
        [0., 0.]])

---
## 7. Storage Buffers (`torch.Storage`)

Inspect the physical 1D contiguous storage buffer underlying the tensor view.

In [23]:
# 7.1 Inspecting and mutating underlying storage
points = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
storage = points.untyped_storage()

print("Points Tensor (3x2):\n", points)
print(f"Physical 1D Storage length: {len(storage)} elements / bytes")
print("Storage raw byte values:", [storage[i] for i in range(min(len(storage), 24))])

# Mutating storage directly reflects in the 2D tensor view
storage[0] = 99
print("\nPoints Tensor after mutating storage[0] = 99:\n", points)

Points Tensor (3x2):
 tensor([[1., 2.],
        [3., 4.],
        [5., 6.]])
Physical 1D Storage length: 24 elements / bytes
Storage raw byte values: [0, 0, 128, 63, 0, 0, 0, 64, 0, 0, 64, 64, 0, 0, 128, 64, 0, 0, 160, 64, 0, 0, 192, 64]

Points Tensor after mutating storage[0] = 99:
 tensor([[1.0000, 2.0000],
        [3.0000, 4.0000],
        [5.0000, 6.0000]])

---
## 8. Stride Mathematics, Transposition & Contiguity

Compute physical offsets: $\text{Offset} = \text{offset} + \sum i_k \cdot \text{stride}[k]$, and manage memory contiguity.

In [24]:
# 8.1 Strides, Slicing, and Transposition
points = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
print(f"Original points -> shape: {points.shape}, stride: {points.stride()}, offset: {points.storage_offset()}")

# Slice sub-tensor (second row)
second_pt = points[1]
print(f"second_pt -> shape: {second_pt.shape}, stride: {second_pt.stride()}, offset: {second_pt.storage_offset()}")
assert points.untyped_storage().data_ptr() == second_pt.untyped_storage().data_ptr(), "Storage pointer mismatch"

# Transpose matrix: swaps strides with zero data copying
points_t = points.t()
print(f"\nTransposed points_t -> shape: {points_t.shape}, stride: {points_t.stride()}, contiguous: {points_t.is_contiguous()}")

# Contiguity resolution for operations like view
try:
    points_t.view(6)
except RuntimeError as e:
    print(f"Expected view error on non-contiguous tensor: {e}")

points_t_cont = points_t.contiguous()
print(f"points_t_cont -> contiguous: {points_t_cont.is_contiguous()}, stride: {points_t_cont.stride()}")
print(f"points_t_cont.view(6) successfully produces: {points_t_cont.view(6)}")

Original points -> shape: torch.Size([3, 2]), stride: (2, 1), offset: 0
second_pt -> shape: torch.Size([2]), stride: (1,), offset: 2
Transposed points_t -> shape: torch.Size([2, 3]), stride: (1, 2), contiguous: False
Expected view error on non-contiguous tensor: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.points_t_cont -> contiguous: True, stride: (3, 1)
points_t_cont.view(6) successfully produces: tensor([1., 3., 5., 2., 4., 6.])

---
## 9. Low-Level Memory Manipulation with `as_strided`

Surgically construct sliding window views without duplicating memory.

In [25]:
# 9.1 Sliding Window View via as_strided
base_vec = torch.arange(10, dtype=torch.float32)
# Window size = 4, step = 1 across 10 elements -> 7 windows
windows = base_vec.as_strided(size=(7, 4), stride=(1, 1), storage_offset=0)

print("Base 1D Tensor:", base_vec)
print("7 Sliding Windows (size=4, step=1):\n", windows)
assert windows.shape == (7, 4), "Window shape mismatch" 

Base 1D Tensor: tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])
7 Sliding Windows (size=4, step=1):
 tensor([[0., 1., 2., 3.],
        [1., 2., 3., 4.],
        [2., 3., 4., 5.],
        [3., 4., 5., 6.],
        [4., 5., 6., 7.],
        [5., 6., 7., 8.],
        [6., 7., 8., 9.]])

---
## 10. Hardware Device Management (CPU $\leftrightarrow$ GPU CUDA)

Move tensors across the PCIe bus, monitor VRAM allocation, and enforce device matching.

In [26]:
# 10.1 Host-Device Transfers and CUDA Memory
cpu_t = torch.randn(2000, 2000)
gpu_t = cpu_t.to(device=device)

print(f"CPU Tensor Device: {cpu_t.device}")
print(f"GPU Tensor Device: {gpu_t.device}")

if device.type == 'cuda':
    print(f"VRAM Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB")
    print(f"VRAM Reserved: {torch.cuda.memory_reserved() / 1e6:.2f} MB")

# GPU computation
gpu_result = torch.matmul(gpu_t, gpu_t)
print(f"GPU Matmul complete: {gpu_result.shape} on {gpu_result.device}")

CPU Tensor Device: cpu
GPU Tensor Device: cuda:0
VRAM Allocated: 16.78 MB
VRAM Reserved: 16.78 MBGPU Matmul complete: torch.Size([2000, 2000]) on cuda:0

---
## 11. NumPy Interoperability (Zero-Copy Shared Memory)

Convert between PyTorch and NumPy with zero memory duplication on CPU.

In [27]:
# 11.1 Zero-Copy Buffer Sharing
torch_vec = torch.ones(4, dtype=torch.float32)
np_view = torch_vec.numpy()

print("Original NumPy View:", np_view)
torch_vec.add_(10.0)
print("NumPy View after PyTorch In-Place Mutation (+10):", np_view)
assert np_view[0] == 11.0, "Zero-copy shared memory failed"

# From NumPy to PyTorch
np_source = np.array([100.0, 200.0, 300.0], dtype=np.float32)
torch_target = torch.from_numpy(np_source)
print("\nPyTorch Tensor from NumPy:", torch_target)

Original NumPy View: [1. 1. 1. 1.]
NumPy View after PyTorch In-Place Mutation (+10): [11. 11. 11. 11.]

PyTorch Tensor from NumPy: tensor([100., 200., 300.])

---
## 12. Generalized Tensors: Sparse Coordinates (COO)

Compress large matrices with few non-zero entries.

In [28]:
# 12.1 Sparse COO Tensor
indices = torch.tensor([[0, 1, 2], [2, 0, 1]], dtype=torch.int64)
values = torch.tensor([3.0, 4.0, 5.0], dtype=torch.float32)
sparse_mat = torch.sparse_coo_tensor(indices, values, (1000, 1000))

print(f"Sparse Matrix Shape: {sparse_mat.shape}")
print(f"Non-Zero Count (_nnz): {sparse_mat._nnz()}")
dense_equivalent_mb = (1000 * 1000 * 4) / 1e6
sparse_actual_bytes = indices.numel() * 8 + values.numel() * 4
print(f"Dense Size: {dense_equivalent_mb:.2f} MB vs. Sparse Size: {sparse_actual_bytes} Bytes")

Sparse Matrix Shape: torch.Size([1000, 1000])
Non-Zero Count (_nnz): 3
Dense Size: 4.00 MB vs. Sparse Size: 60 Bytes

C:\Users\emrea\AppData\Local\Temp\ipykernel_1556\1301952109.py:4: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:823.)
  sparse_mat = torch.sparse_coo_tensor(indices, values, (1000, 1000))

---
## 13. Serialization: PyTorch Checkpoints & HDF5 (`h5py`)

Persist weights to disk and stream chunked multidimensional arrays.

In [29]:
# 13.1 PyTorch Native Checkpoint Serialization
ckpt = {'weights': torch.randn(3, 3), 'epoch': 5}
torch.save(ckpt, 'sample_ckpt.pt')
loaded_ckpt = torch.load('sample_ckpt.pt', weights_only=True)
print("Loaded Checkpoint Keys:", list(loaded_ckpt.keys()))
os.remove('sample_ckpt.pt')

# 13.2 Chunked HDF5 Storage
data_matrix = torch.arange(100, dtype=torch.float32).reshape(10, 10)
with h5py.File('sample_data.h5', 'w') as h5f:
    h5f.create_dataset('matrix', data=data_matrix.numpy())

with h5py.File('sample_data.h5', 'r') as h5f:
    streamed_sub_tensor = torch.from_numpy(h5f['matrix'][2:6, :])
    print("Streamed HDF5 Slice (rows 2..6) Shape:", streamed_sub_tensor.shape)
os.remove('sample_data.h5')

Loaded Checkpoint Keys: ['weights', 'epoch']
Streamed HDF5 Slice (rows 2..6) Shape: torch.Size([4, 10])

---
## 14. Chapter 3 Exercises Solutions

Analytical and programmatic verification of Section 3.15 exercises.

In [30]:
# Exercise 1: Storage, view, and offset of range(9)
a = torch.tensor(list(range(9)))
print(f"Exercise 1.a -> a: size={a.size()}, offset={a.storage_offset()}, stride={a.stride()}")

b = a.view(3, 3)
print(f"Exercise 1.a -> b: size={b.size()}, offset={b.storage_offset()}, stride={b.stride()}")
print("Do a and b share storage?", a.untyped_storage().data_ptr() == b.untyped_storage().data_ptr())

c = b[1:, 1:]
print(f"Exercise 1.b -> c: size={c.size()}, offset={c.storage_offset()}, stride={c.stride()}")
print("Tensor c values:\n", c)

# Exercise 2: Mathematical operations and in-place semantics
int_t = torch.tensor([1, 4, 9, 16], dtype=torch.int32)
try:
    int_t.sqrt_()
except RuntimeError as e:
    print(f"\nExercise 2 -> In-place sqrt on int32 failed as expected: {e}")

float_t = int_t.to(dtype=torch.float32)
float_t.sqrt_()
print(f"Exercise 2 -> Successful float32 in-place sqrt: {float_t}")

Exercise 1.a -> a: size=torch.Size([9]), offset=0, stride=(1,)
Exercise 1.a -> b: size=torch.Size([3, 3]), offset=0, stride=(3, 1)
Do a and b share storage? True
Exercise 1.b -> c: size=torch.Size([2, 2]), offset=4, stride=(3, 1)
Tensor c values:
 tensor([[4, 5],
        [7, 8]])
Exercise 2 -> In-place sqrt on int32 failed as expected: result type Float can't be cast to the desired output type Int
Exercise 2 -> Successful float32 in-place sqrt: tensor([1., 2., 3., 4.])